<a href="https://colab.research.google.com/github/maryorimontoya10/Proyecto-Marketplace/blob/main/ProyectoBootcamp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Tendencias de compra Marketplace**

**Librerías**


In [ ]:
#Librería para generar PDF
!pip install reportlab

In [ ]:
import pandas as pd #DataFrame
import matplotlib.pyplot as plt #Gráficos
import seaborn as sns #Gráficos

# Para generar el archivo .xlsx
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.drawing.image import Image as XLImage
from openpyxl.worksheet.table import Table as XLTable, TableStyleInfo

#Para generar el archivo PDF
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors

# Administrar y organizar los archivos de salida
import os

**Importación base de datos**

In [ ]:
from google.colab import files

# OPCIÓN 1: Por si se quiere subir el archivo desde el PC
# uploaded = files.upload() #Subir desde el pc
# archivo = 'BD_Marketplace.xlsx'

# OPCIÓN 2: Lectura directa desde GitHub
archivo = 'https://github.com/maryorimontoya10/Proyecto-Marketplace/blob/main/BD_Marketplace.xlsx?raw=true'

tipos_esperados = {
    'Nro del pedido': str,
    'Nro ident.fis.1': str,
    'Vendedor': str,
    'Nro documento': str,
    'Comision sin IVA': float,
    'PLU': str,
    'Codigo EAN': str,
    'Denominacion': str,
    'Importe': float,
    'Cantidad de pedido': float,
    'Tipo de despacho': str,
    'MES': str,
    'CONCEPTO': str,
    'FACTURA': str,
    'COSTO': float,
    'FLETE': float,
    'Envios exito': float,
    'Mes Facturacion': str,
    'Recaudo': float,
    'Gen': str,
    'Subdireccion': str,
    'Sublinea E-commerce': str,
    'Sublinea': str,
    'Sublinea Especifica': str,
    'Tipologia producto': str,
    'Ejecutivo comercial': str
}

columnas_fecha = ['Fecha','FECHA FACTURA']

try:
    df = pd.read_excel(
        archivo,
        dtype=tipos_esperados,
        parse_dates=columnas_fecha,
        sheet_name='Comision2025'
    )
    print("¡Importación exitosa!")
    df.info()

except FileNotFoundError:
    print(f"El archivo '{archivo}' no existe. Asegúrate de subirlo.")
except Exception as e:
    print(f"Error al leer el archivo: {e}")

Limpieza y validación de datos

In [ ]:
#Tratamiento de variables numéricas
columnas_numericas = ['Comision sin IVA', 'Importe', 'Cantidad de pedido', 'COSTO', 'FLETE', 'Envios exito', 'Recaudo']
df[columnas_numericas]=df[columnas_numericas].apply(pd.to_numeric,errors='coerce')
df = df.dropna(subset=columnas_numericas)

for col in columnas_numericas:
    if col not in df.columns:
      print("La columna", col, "no existe en el DataFrame.")
      exit()

columnas_a_filtrar_negativos = ['Comision sin IVA', 'COSTO', 'Importe', 'Recaudo']
df_filtrado = df[(df[columnas_a_filtrar_negativos] >= 0).all(axis=1)]

print("Filas con valores negativos:",(df[columnas_a_filtrar_negativos] < 0).any(axis=1).sum())
print("\n")

#Tratamiento de nulos en variables Categóricas
columnas_categoricas_clave = [
    'Vendedor',
    'Ejecutivo comercial',
    'Gen',
    'Tipologia producto',
    'Denominacion'
]
df[columnas_categoricas_clave] = df[columnas_categoricas_clave].fillna('SIN ASIGNAR')


print(f"Número de valores 'No definido' después de la imputación:")
for col in columnas_categoricas_clave:
    conteo = (df[col] == 'No definido').sum()
    if conteo > 0:
        print(f"  {col}: {conteo} imputaciones")
    else:
        print(f"  {col}: No se encontraron nulos originales.")
print("\n")

#Tratamiento de nulos en fechas clave

nulos_fecha = df['Fecha'].isna().sum()

if nulos_fecha > 0:
    df_limpio = df.dropna(subset=['Fecha']).copy()

    print(f"Se encontraron {nulos_fecha} filas con la 'Fecha' nula.")
    print(f"Estas filas fueron eliminadas. Tamaño del DataFrame antes: {len(df)}, después: {len(df_limpio)}")

    df = df_limpio
else:
    print("No se encontraron filas con la 'Fecha' nula. El DataFrame está listo para el análisis temporal.")

#**Objetivo específico 1: Identificar los productos y vendedores de más alto rendimiento.**

#¿Cuáles son los productos más vendidos en cantidad y valor?

In [ ]:
COLUMNAS_PRODUCTO = ['PLU', 'Denominacion']

df_resumen_ventas = df.groupby(COLUMNAS_PRODUCTO).agg(
    Valor_Total_Vendido=('COSTO', 'sum')
).reset_index()

print("TOP 10 Productos por Valor Total Vendido (Ingreso)")
top_valor = df_resumen_ventas.sort_values(
    by='Valor_Total_Vendido',
    ascending=False
).head(10)

display(
    top_valor.style.format({
        'Valor_Total_Vendido': '{:,.0f}'
    })
)

# Gráfico de barras
df_barras_valor_top10 = top_valor.copy()
df_barras_valor_top10 = df_barras_valor_top10.sort_values(
    by='Valor_Total_Vendido',
    ascending=False
)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Denominacion',
    y='Valor_Total_Vendido',
    data=df_barras_valor_top10,
    hue='Denominacion',
    legend=False,
    palette='viridis'
)

plt.title('Top 10 Productos por Valor Total Vendido', fontsize=16)
plt.xlabel('Denominación del Producto')
plt.ylabel('Valor Total Vendido ($)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("Productos por Valor Total Vendido.png", dpi=300, bbox_inches="tight")
plt.show()

#¿Qué vendedores generan mayor rentabilidad en la comisión?


In [ ]:
df_mejores_vendedores = df.groupby('Vendedor').agg(
    Comision_Total=('Comision sin IVA', 'sum')
).reset_index()

print("TOP 10 Vendedores por Comision sin IVA")
top_vendedores = df_mejores_vendedores.sort_values(
    by='Comision_Total',
    ascending=False
).head(10)
display(
    top_vendedores.style.format({
        'Comision_Total': '{:,.0f}'
    })
)

#Gráfico de barras
df_barras_comision_top10 = top_vendedores.copy()
df_barras_comision_top10 = df_barras_comision_top10.sort_values(
    by='Comision_Total',
    ascending=False
)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Vendedor',
    y='Comision_Total',
    data=df_barras_comision_top10,
    hue='Vendedor',
    legend=False,
    palette='magma'
)

plt.title('Top 10 Vendedores por Comisión Total Generada', fontsize=16)
plt.xlabel('Vendedor')
plt.ylabel('Comisión Total ($)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("Vendedores por Comisión Total Generada.png", dpi=300, bbox_inches="tight")
plt.show()

#¿Qué categorías de productos están generando mayor venta y estancamiento de la misma?


In [ ]:
CATEGORIAS_PRODUCTO = ['Gen', 'Tipologia producto']

df_ventas_productos = df.groupby(CATEGORIAS_PRODUCTO).agg(
    Total_Vendido=('COSTO', 'sum')
).reset_index()

print("TOP 10 Productos más vendidos por Categoria")
top_categorias = df_ventas_productos.sort_values(
    by='Total_Vendido',
    ascending=False
    ).head(10)
display(
    top_categorias.style.format({
        'Total_Vendido': '{:,.0f}'
    })
)


print("\n")

print("TOP 10 Productos menos Vendidos por Categoria")
top_categorias = df_ventas_productos.sort_values(
    by='Total_Vendido'
    ).head(10)
display(
    top_categorias.style.format({
        'Total_Vendido': '{:,.0f}'
    })
)

#Gráfico de torta
top_categorias_plot = df_ventas_productos.sort_values(
    by='Total_Vendido',
    ascending=False
).head(10).copy()

top_categorias_plot['Categoria_Combinada'] = top_categorias_plot['Gen'] + ' - ' + top_categorias_plot['Tipologia producto']

etiquetas_categorias = top_categorias_plot['Categoria_Combinada']
datos_categorias = top_categorias_plot['Total_Vendido']

plt.figure(figsize=(12, 12))

plt.pie(
    datos_categorias,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.85,
    wedgeprops={'edgecolor': 'black', 'linewidth': 1}
)

plt.legend(
    etiquetas_categorias,
    title="Categoría de Producto",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

plt.title('Contribución Porcentual del Top 10 de Categorías a la Venta Total', fontsize=16)
plt.tight_layout()
plt.savefig("Contribución Porcentual de Categorías a la Venta Total.png", dpi=300, bbox_inches="tight")
plt.show()

#**Objetivo específico 2: Analizar la tendencia de compras según la temporada del año.**

#¿Cómo se comportan las ventas según la temporada del año?


In [ ]:
if 'Mes_Nombre' not in df.columns:
    df['Mes_Num'] = df['Fecha'].dt.month
    df['Mes_Nombre'] = df['Fecha'].dt.strftime('%b')

CATEGORIAS_PRODUCTO = ['Gen', 'Tipologia producto']

analisis_costos_estacional = df.groupby(['Mes_Num', 'Mes_Nombre']).agg(
    Costo_Total_Acumulado=('COSTO', 'sum')
).reset_index()

analisis_costos_estacional = analisis_costos_estacional.sort_values(by='Mes_Num')

df_tendencia_nombre_mes = df.groupby(['Mes_Num', 'Mes_Nombre'] + CATEGORIAS_PRODUCTO).agg(
    Costo_Mensual=('COSTO', 'sum')
).reset_index()

top_5_categorias = top_categorias_plot['Tipologia producto'].head(5).tolist()

df_tendencia_top5_nombre_mes = df_tendencia_nombre_mes[
    df_tendencia_nombre_mes['Tipologia producto'].isin(top_5_categorias)
].copy()

df_tendencia_top5_nombre_mes.sort_values(by='Mes_Num', inplace=True)

display(
    df_tendencia_top5_nombre_mes.head(10).style.format({
        'Costo_Mensual': '{:,.0f}'
    })
)

#Gráfico de barras
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Mes_Nombre',
    y='Costo_Total_Acumulado',
    data=analisis_costos_estacional,
    hue='Mes_Nombre',
    legend=False,
    palette='viridis'
)

plt.title('Venta Total por Mes (Estacionalidad)', fontsize=16)
plt.xlabel('Mes del Año')
plt.ylabel('Costo Total ($)')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("Venta Total por Mes.png", dpi=300, bbox_inches="tight")
plt.show()

#Gráfico de lineas
plt.figure(figsize=(14, 7))
sns.lineplot(
    x='Mes_Nombre',
    y='Costo_Mensual',
    hue='Tipologia producto',
    data=df_tendencia_top5_nombre_mes,
    marker='o'
    )

plt.title('Tendencia Mensual del Costo Vendido para el Top 5 de Categorías', fontsize=16)
plt.xlabel('Mes')
plt.ylabel('Costo Total Vendido Mensual ($)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Tipología Producto', bbox_to_anchor=(1.05, 1), loc=2)
plt.tight_layout()
plt.savefig("Tendencia Mensual del Costo Vendido.png", dpi=300, bbox_inches="tight")
plt.show()

#**Objetivo específico 3: Evaluar la eficiencia del equipo comercial.**

#¿Cuál es la comisión por ejecutivo?

In [ ]:
df_comision_ejecutivo = df.groupby('Ejecutivo comercial').agg(
    Comision_Total=('Comision sin IVA', 'sum'),
    Num_Vendedores_Que_Supervisa=('Vendedor', 'nunique')
).reset_index()

df_comision_ejecutivo = df_comision_ejecutivo.sort_values(
    by='Comision_Total',
    ascending=False
)

print("TOP Ejecutivos por Comisión Total")
display(
    df_comision_ejecutivo.head(10).style.format({
        'Comision_Total': '{:,.0f}',
        'Num_Vendedores_Que_Supervisa': '{:.0f}'
    })
)

#Gráfica
plt.figure(figsize=(12, 8))
sns.barplot(
    x='Ejecutivo comercial',
    y='Comision_Total',
    data=df_comision_ejecutivo.head(10),
    hue='Ejecutivo comercial',
    legend=False,
    palette='Spectral'
)
plt.title('Top 10 Ejecutivos por Comisión Total Generada', fontsize=16)
plt.xlabel('Ejecutivo Comercial')
plt.ylabel('Comisión Total ($)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("Ejecutivos por Comisión Total Generada.png", dpi=300, bbox_inches="tight")
plt.show()

#¿Cuál es la variabilidad de la comisión por temporada de cada ejecutivo?

In [ ]:
COLUMNA_COMISION = 'Comision sin IVA'

df['Métrica_Comision_Individual'] = df[COLUMNA_COMISION]

df_estadisticas_boxplot = df.groupby('Ejecutivo comercial')['Métrica_Comision_Individual'].describe().reset_index()

df_estadisticas_boxplot = df_estadisticas_boxplot.rename(columns={'50%': 'Mediana (Q2)'})

print("Estadísticas Clave del Boxplot (Comisión Individual por Ejecutivo)")
display(
    df_estadisticas_boxplot.style.format({
        'count': '{:.0f}',          # Conteo como entero
        'mean': '{:,.0f}',           # Media sin decimales
        'std': '{:,.0f}',            # Desviación estándar
        'min': '{:,.0f}',            # Mínimo
        '25%': '{:,.0f}',            # Primer Cuartil (Q1)
        'Mediana (Q2)': '{:,.0f}',   # Mediana (Q2)
        '75%': '{:,.0f}',            # Tercer Cuartil (Q3)
        'max': '{:,.0f}'             # Máximo
    })
)

#Box plot
plt.figure(figsize=(16, 8))

sns.boxplot(
    x='Ejecutivo comercial',
    y='Métrica_Comision_Individual',
    data=df,
    hue='Ejecutivo comercial',
    legend=False,
    palette='Spectral'
)

plt.title('Variabilidad y Consistencia de la Comisión Individual por Ejecutivo', fontsize=18)
plt.xlabel('Ejecutivo Comercial')
plt.ylabel('Comisión Individual por Venta ($)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("Boxplot_Variabilidad_Comision.png", dpi=300, bbox_inches="tight")
plt.show()

#**Exportación a Excel**

In [ ]:
ruta_excel = "Resultados_Marketplace.xlsx"
wb = Workbook()

# Eliminar hoja inicial vacía
wb.remove(wb.active)

elementos = [
    ("Prod Valor Vendido", top_valor, "Productos por Valor Total Vendido.png"),
    ("Vend Comision Total", top_vendedores, "Vendedores por Comisión Total Generada.png"),
    ("Cat Venta Total", top_categorias_plot, "Contribución Porcentual de Categorías a la Venta Total.png"),
    ("Venta Total Mes", analisis_costos_estacional, "Venta Total por Mes.png"),
    ("Tendencia Costo Vendido", df_tendencia_top5_nombre_mes, "Tendencia Mensual del Costo Vendido.png"),
    ("Ejecutivos Comision", df_comision_ejecutivo, "Ejecutivos por Comisión Total Generada.png"),
    ("Boxplot Variabilidad Comision", df_estadisticas_boxplot, "Boxplot_Variabilidad_Comision.png")
]


# Crear hojas con tabla + gráfica
for nombre_hoja, df_export, grafica in elementos:

    ws = wb.create_sheet(nombre_hoja)

    # Insertar tabla
    for fila in dataframe_to_rows(df_export, index=False, header=True):
        ws.append(fila)

    # Rango de tabla dinámico
    max_fila = ws.max_row
    max_col = ws.max_column
    rango = f"A1:{chr(64+max_col)}{max_fila}"

# Genere un nombre de tabla válido reemplazando espacios con guiones bajos
    table_name = nombre_hoja.replace(" ", "_")
    tabla = XLTable(ref=rango, displayName=f"Tabla_{table_name}")
    tabla.name = f"Tabla_{table_name}"

    estilo = TableStyleInfo(
        name="TableStyleMedium9",
        showRowStripes=True,
        showColumnStripes=False
    )
    tabla.tableStyleInfo = estilo
    ws.add_table(tabla)

    # FORMATO DE MILES EN EXCEL
    for col in ws.iter_cols(min_row=2, max_row=ws.max_row):
        for cell in col:
            if isinstance(cell.value, (int, float)):
                cell.number_format = '#,##0'

    # Insertar gráfica
    try:
        img = XLImage(grafica)
        img.width = 550
        img.height = 350
        ws.add_image(img, f"{chr(66+max_col)}2")  # Se pone al lado de la tabla
    except:
        print(f"No se encontró la imagen {grafica}")

# DASHBOARD

ws_dash = wb.create_sheet("Dashboard")

ventas_totales = df["COSTO"].sum()
comision_total = df["Comision sin IVA"].sum()
transacciones = df.shape[0]

# KPIs
ws_dash["A1"] = "Indicadores clave"
ws_dash["A3"] = "Ventas Totales:"
ws_dash["B3"] = float(ventas_totales)

ws_dash["A4"] = "Comisión Total:"
ws_dash["B4"] = float(comision_total)

ws_dash["A5"] = "Número de Transacciones:"
ws_dash["B5"] = int(transacciones)

ws_dash["B3"].number_format = '#,##0'
ws_dash["B4"].number_format = '#,##0'
ws_dash["B5"].number_format = '#,##0'

fila_img = 20
imagenes_dashboard = [
    "Productos por Valor Total Vendido.png",
    "Vendedores por Comisión Total Generada.png",
    "Contribución Porcentual de Categorías a la Venta Total.png",
    "Venta Total por Mes.png",
    "Tendencia Mensual del Costo Vendido.png",
    "Ejecutivos por Comisión Total Generada.png",
    "Boxplot_Variabilidad_Comision.png"
]

# Gráficas en el dashboard
for img_name in imagenes_dashboard:
    try:
        img = XLImage(img_name)
        img.width = 550
        img.height = 350
        ws_dash.add_image(img, f"A{fila_img}")
        fila_img += 20
    except:
        print(f"No se encontró {img_name}")

# Guardar Excel
wb.save(ruta_excel)
print("Excel final exportado:", ruta_excel)

#**Exporte PDF**


In [ ]:
ruta_pdf = "Resumen_Analisis_Marketplace.pdf"
styles = getSampleStyleSheet()
story = []

doc = SimpleDocTemplate(ruta_pdf, pagesize=letter)
doc.title = "Dashboard Marketplace"

def df_to_pdf_table(df):
    tabla = [df.columns.tolist()]
    for row in df.values:
        nueva_fila = []
        for val in row:
            if isinstance(val, (int, float)):
                nueva_fila.append(f"{val:,.0f}")
            else:
                nueva_fila.append(str(val))
        tabla.append(nueva_fila)
    return tabla

# Título
story.append(Paragraph("Resumen Análisis Marketplace", styles["Title"]))
story.append(Spacer(1, 20))

# KPIs en tabla
kpi_data = [
    ["Indicador", "Valor"],
    ["Ventas Totales", f"{ventas_totales:,.0f}"],
    ["Comisión Total", f"{comision_total:,.0f}"],
    ["Número Transacciones", f"{transacciones:,.0f}"]
]

tabla_kpi = Table(kpi_data, colWidths=[150, 200])
tabla_kpi.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#444444")),
    ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("GRID", (0,0), (-1,-1), 0.5, colors.black),
    ("FONT", (0,0), (-1,-1), "Helvetica", 10),
    ("ALIGN", (1,1), (-1,-1), "CENTER")
]))
story.append(tabla_kpi)
story.append(Spacer(1, 20))

# Gráficas
story.append(Paragraph("Gráficas Principales", styles["Heading2"]))
story.append(Spacer(1, 10))

for imagen in imagenes_dashboard:
    if os.path.exists(imagen):
        story.append(Image(imagen, width=480, height=300))
        story.append(Spacer(1, 20))

# Crear PDF
doc.build(story)
print("PDF generado:", ruta_pdf)